# Lip-sync (LatentSync) — Colab entrypoint

Runs `gpu/lipsync_latentsync.py` as an importable function. Defaults to
**LatentSync 1.5** (256px, ~8 GB VRAM) — fits a free T4's 16 GB with room to
spare; switch to `resolution=512` for LatentSync 1.6 if you want sharper
output and don't mind the extra VRAM and runtime.

Face detection defaults to a MediaPipe-based detector
(`gpu/patches/mediapipe_face_detector.py`), not LatentSync's stock InsightFace
one — InsightFace's pretrained weights are non-commercial only. See
`docs/licenses.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
REPO_URL = "<your fork/clone URL>"
REPO_DIR = "/content/presenter-video"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/presenter-video'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['PRESENTER_DRIVE_DIR'] = DRIVE_DIR
os.environ['PRESENTER_WEIGHTS_DIR'] = f'{DRIVE_DIR}/weights'   # cache the multi-GB checkpoint on Drive
os.environ['HF_HOME'] = f'{DRIVE_DIR}/weights/hf'

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements/lipsync.txt
!apt -y install libgl1 > /dev/null

In [ ]:
# Vendors LatentSync into vendor/LatentSync at the pinned commit, if not already present.
!bash scripts/vendor_latentsync.sh

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from gpu.lipsync_latentsync import run

presenter_clip = f"{REPO_DIR}/assets/samples/placeholder_presenter.mp4"  # swap for a real, consented clip
voice_audio = f"{DRIVE_DIR}/voice_test.wav"                              # output of the voice notebook
out_path = f"{DRIVE_DIR}/lipsync_test.mp4"

run(
    presenter_clip, voice_audio, out_path,
    device="cuda", resolution=256,
    weights_dir=f"{DRIVE_DIR}/weights",
)
print("wrote", out_path)

In [ ]:
from IPython.display import Video
Video(out_path, embed=True, width=360)